# 🏥 ViBioMIR 2026 - Qdrant Int8 Quantized Hybrid Retrieval Pipeline

Notebook chính thức chạy trọn gói trên Kaggle (GPU 2x T4 hoặc L4):
- **Qdrant Embedded on-disk** + **Scalar Quantization Int8** (tiết kiệm 4x bộ nhớ, RAM < 2GB)
- **Tái sử dụng 1.63M Dense vectors** từ FAISS (0 giây re-encode GPU)
- **Dual-GPU Allocation**: BGE-M3 trên `cuda:0`, BGE-Reranker-Large trên `cuda:1`
- **Qdrant Score Fusion**: Giữ trọn độ nhạy từ khóa y khoa
- **Smart Chunk Selector**: Tối ưu hóa F2-Macro (Top 35 docs, Top 45 chunks, threshold -2.5)


### Bước 1: Cài đặt toàn bộ thư viện phụ thuộc

In [ ]:
!pip install -q "huggingface_hub[cli]" qdrant-client FlagEmbedding faiss-cpu bm25s pyvi jieba loguru pydantic rank-bm25 trafilatura ijson httpx sentence-transformers accelerate rich typer sacremoses

### Bước 2: Kiểm tra thư viện đã nạp thành công

In [ ]:
import torch
import qdrant_client
import FlagEmbedding
import faiss
import loguru

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print("✅ Toàn bộ thư viện đã sẵn sàng!")

### Bước 3: Đăng nhập Hugging Face & Clone Repo mới nhất

In [ ]:
import os
import subprocess

# Thay token của bạn vào (hoặc lấy từ Kaggle Secrets)
HF_TOKEN = os.getenv("HF_TOKEN", "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
subprocess.run(f"hf auth login --token {HF_TOKEN}", shell=True, check=True)

# Clone repo nếu chưa có, hoặc pull code mới nhất
if not os.path.exists("/kaggle/working/med-doc-retrieval"):
    !git clone https://github.com/nieths/med-doc-retrieval.git /kaggle/working/med-doc-retrieval

%cd /kaggle/working/med-doc-retrieval
!git pull origin main

### Bước 4: Tải dữ liệu và Indices từ Hugging Face Bucket

In [ ]:
%%bash
cd /kaggle/working/med-doc-retrieval
mkdir -p data/indices data/processed data/raw/vibio_mir

# 1. Kéo metadata SQLite (2.4 GB - 1.63M chunks)
hf cp hf://buckets/nieths/ViBioMIR/indices/chunks_meta.sqlite data/indices/chunks_meta.sqlite

# 2. Kéo Dense Index FAISS (6.3 GB - 1.63M dense vectors để nạp thẳng vào Qdrant)
hf cp hf://buckets/nieths/ViBioMIR/indices/dense_index.faiss data/indices/dense_index.faiss

# 3. Kéo Cache queries đã làm giàu & query.parquet
hf cp hf://buckets/nieths/ViBioMIR/processed/queries_enriched.jsonl data/processed/queries_enriched.jsonl
hf download AIGuruTinix/ViBioMIR query.parquet --repo-type dataset --local-dir data/raw/vibio_mir/

# Kiểm tra dung lượng file tải về
ls -lh data/indices/
ls -lh data/processed/

### Bước 5: Chạy thử nghiệm nhanh Sanity Test (5,000 chunks)

In [ ]:
%cd /kaggle/working/med-doc-retrieval
!python3 scripts/run_qdrant_int8_pipeline.py \n    --build \n    --infer \n    --max-chunks 5000 \n    --batch-size 1000

### Bước 6: Chạy chính thức toàn bộ 1.63 Triệu Chunks & Tạo Submission

In [ ]:
%cd /kaggle/working/med-doc-retrieval
!python3 scripts/run_qdrant_int8_pipeline.py \n    --build \n    --infer \n    --batch-size 2000 \n    --top-k-docs 35 \n    --top-k-chunks 45 \n    --score-threshold -2.5

### Bước 7: Kiểm tra file Submission sẵn sàng nộp

In [ ]:
import os
from pathlib import Path

zip_file = Path("/kaggle/working/submission.zip")
if not zip_file.exists():
    zip_file = Path("/kaggle/working/med-doc-retrieval/results/submission.zip")

if zip_file.exists():
    size_mb = zip_file.stat().st_size / (1024 * 1024)
    print(f"🎉 THÀNH CÔNG! File nộp bài: {zip_file} ({size_mb:.2f} MB)")
else:
    print("❌ Chưa tìm thấy file submission.zip, vui lòng kiểm tra log chạy ở Bước 6!")